# Industrial Machine Condition Classification
## Gradient Boosting (LightGBM) with Feature Engineering

**Target:** Predict `condition` (0 = normal, 1 = degraded, 2 = critical)  
**Approach:** Feature engineering → LightGBM with cross-validation → submission

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42

## 1. Load Data

In [2]:
train = pd.read_csv('data/student_dataset.csv')
test  = pd.read_csv('data/test.csv')
sub   = pd.read_csv('data/sample_submission.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head()

Train: (16000, 13)  |  Test: (4000, 12)


,unit_id,unit_type,ambient_temp,process_temp,coolant_flow,shaft_speed,drive_torque,power_load,component_wear,thermal_stress,mechanical_load,vibration_index,condition
0,UNT-184174,M,21.5,56.4,17.85,1454.0,58.2,9.099,30.0,29.31,57.19,1.768,0
1,UNT-236234,H,29.9,65.0,30.88,1720.0,54.6,10.101,13.0,23.93,36.94,1.081,0
2,UNT-247460,M,26.4,47.8,19.75,1552.0,38.4,6.328,89.0,13.52,38.22,1.031,0
3,UNT-176316,M,28.8,60.9,19.10,1513.0,49.1,7.768,59.0,24.62,46.17,1.011,0
4,UNT-105110,S,29.4,64.0,11.77,1536.0,52.6,8.405,36.0,27.83,94.72,1.851,1


## 2. EDA

In [3]:
# --- Pairplot of highest-correlation features coloured by condition ---
plot_df = train[['vibration_index', 'component_wear', 'thermal_stress',
                 'mechanical_load', 'condition']].dropna().sample(2000, random_state=SEED)
plot_df['Condition'] = plot_df['condition'].map({0: 'Normal', 1: 'Degraded', 2: 'Critical'})

g = sns.pairplot(plot_df.drop(columns='condition'), hue='Condition',
                 plot_kws={'alpha': 0.4, 's': 15},
                 palette={'Normal': '#4878CF', 'Degraded': '#f7a035', 'Critical': '#d62728'})
g.figure.suptitle('Key Feature Pairplot by Condition', y=1.01)
plt.show()

RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x77aa76f59bc0> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 1000x1000 with 16 Axes>

In [4]:
# --- Condition × unit_type heatmap ---
ct = pd.crosstab(train['unit_type'], train['condition'])
ct.columns = ['Normal', 'Degraded', 'Critical']

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Condition Count by Unit Type')
ax.set_xlabel('Condition')
plt.tight_layout()
plt.show()

RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x77aa76f59bc0> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 600x300 with 2 Axes>

In [ ]:
# --- Violin plots for top features ---
top_feats = ['vibration_index', 'component_wear', 'thermal_stress', 'mechanical_load']
plot_df2 = train[top_feats + ['condition']].dropna().copy()
plot_df2['Condition'] = plot_df2['condition'].map({0: 'Normal', 1: 'Degraded', 2: 'Critical'})

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, top_feats):
    sns.violinplot(data=plot_df2, x='Condition', y=col,
                   order=['Normal', 'Degraded', 'Critical'],
                   palette='muted', inner='quartile', ax=ax)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')

plt.suptitle('Top Feature Distributions by Condition', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [5]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Ordinal encode unit_type
    df['unit_type_enc'] = df['unit_type'].map({'S': 0, 'M': 1, 'H': 2})

    # Temperature delta
    df['temp_delta'] = df['process_temp'] - df['ambient_temp']

    # Mechanical power proxy
    df['torque_x_speed'] = df['drive_torque'] * df['shaft_speed']

    # Compound wear signal
    df['wear_x_vibration'] = df['component_wear'] * df['vibration_index']

    # Thermal-mechanical stress product
    df['thermal_x_mech'] = df['thermal_stress'] * df['mechanical_load']

    # Coolant efficiency
    df['heat_per_coolant'] = df['thermal_stress'] / (df['coolant_flow'] + 1e-6)

    # Wear relative to shaft speed
    df['wear_per_speed'] = df['component_wear'] / (df['shaft_speed'] + 1e-6)

    # Power per mechanical load
    df['power_per_load'] = df['power_load'] / (df['mechanical_load'] + 1e-6)

    # Vibration squared (captures nonlinear wear escalation)
    df['vibration_sq'] = df['vibration_index'] ** 2

    # Wear squared
    df['wear_sq'] = df['component_wear'] ** 2

    # 3-way interaction: wear × vibration × thermal
    df['wear_vib_thermal'] = df['component_wear'] * df['vibration_index'] * df['thermal_stress']

    df = df.drop(columns=['unit_id', 'unit_type'], errors='ignore')
    return df


train_fe = engineer_features(train.drop(columns=['condition']))
y        = train['condition'].values
test_fe  = engineer_features(test)

print('Feature matrix shape:', train_fe.shape)
print('Features:', train_fe.columns.tolist())

Feature matrix shape: (16000, 21)
Features: ['ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed', 'drive_torque', 'power_load', 'component_wear', 'thermal_stress', 'mechanical_load', 'vibration_index', 'unit_type_enc', 'temp_delta', 'torque_x_speed', 'wear_x_vibration', 'thermal_x_mech', 'heat_per_coolant', 'wear_per_speed', 'power_per_load', 'vibration_sq', 'wear_sq', 'wear_vib_thermal']


## 4. Impute Missing Values

In [6]:
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(train_fe)
X_test  = imputer.transform(test_fe)

print('Missing after imputation:', np.isnan(X_train).sum())

Missing after imputation: 0


## 5. Cross-Validation

In [7]:
lgb_params = dict(
    objective='multiclass',
    num_class=3,
    metric='multi_logloss',
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=7,
    min_child_samples=20,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=1,   # keep to 1 to avoid threading conflicts in WSL2
    verbose=-1,
)

# Manual stratified CV (avoids joblib × LightGBM thread collision)
from sklearn.metrics import accuracy_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = []
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y), 1):
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_train[tr_idx], y[tr_idx])
    preds = m.predict(X_train[val_idx])
    acc = accuracy_score(y[val_idx], preds)
    cv_scores.append(acc)
    print(f'  Fold {fold}: {acc:.4f}')

cv_scores = np.array(cv_scores)
print(f'\nCV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

  Fold 1: 0.8859
  Fold 2: 0.8841
  Fold 3: 0.8859
  Fold 4: 0.8841
  Fold 5: 0.8809

CV Accuracy: 0.8842 ± 0.0018


## 6. Train on Full Data & Evaluate

In [8]:
model = lgb.LGBMClassifier(**lgb_params)
model.fit(X_train, y)

train_pred = model.predict(X_train)
print('=== Training Set Report ===')
print(classification_report(y, train_pred, target_names=['Normal', 'Degraded', 'Critical']))

=== Training Set Report ===
              precision    recall  f1-score   support

      Normal       0.99      1.00      0.99      8000
    Degraded       1.00      0.99      1.00      4000
    Critical       1.00      0.98      0.99      4000

    accuracy                           0.99     16000
   macro avg       1.00      0.99      0.99     16000
weighted avg       0.99      0.99      0.99     16000



In [9]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y, train_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Degraded', 'Critical'],
            yticklabels=['Normal', 'Degraded', 'Critical'])
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
ax.set_title('Confusion Matrix (Train)')
plt.tight_layout()
plt.show()

RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x77aa76f59bc0> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 600x500 with 2 Axes>

## 7. Feature Importance

In [10]:
feat_imp = pd.DataFrame({
    'feature':    train_fe.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=feat_imp, x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('LightGBM Feature Importances (Gain)')
ax.set_xlabel('Importance')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print(feat_imp.to_string(index=False))

RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x77aa76f59bc0> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 1000x700 with 1 Axes>

## 8. Generate Submission

In [11]:
test_preds = model.predict(X_test)

submission = pd.DataFrame({
    'unit_id':   test['unit_id'],
    'condition': test_preds
})

submission.to_csv('data/submission_gradient_boosting.csv', index=False)
print('Submission saved.')
print(f'Predicted distribution:\n{pd.Series(test_preds).value_counts().sort_index()}')
submission.head()

Submission saved.
Predicted distribution:
0    2118
1     930
2     952
Name: count, dtype: int64


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0
